In [1]:
import pandas as pd

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', None)

print("Knihovny pripraveny")

Knihovny pripraveny


In [25]:
# 1. Nacteni kaggle dat

df_en = pd.read_csv('../data/raw/medical_transcriptions.csv', index_col=0)

# Pomocná funkce pro načítání českých souborů (řeší kódování a středníky)
def load_cz_csv(file_path):
    try:
        return pd.read_csv(file_path, sep=';', encoding='cp1250', low_memory=False)
    except:
        return pd.read_csv(file_path, sep=';', encoding='utf-8', low_memory=False)

# 1. Načtení Kaggle dat
df_en = pd.read_csv('../data/raw/medical_transcriptions.csv', index_col=0)

# 2. Načtení hlavního registru
df_cz = load_cz_csv('../data/raw/export_2026.csv')

# 3. Načtení služeb
df_services = load_cz_csv('../data/raw/export_sluzby_2026.csv')

In [26]:
print(df_en.head())

                                                         description  \
0   A 23-year-old white female presents with complaint of allergies.   
1                           Consult for laparoscopic gastric bypass.   
2                           Consult for laparoscopic gastric bypass.   
3                                             2-D M-Mode. Doppler.     
4                                                 2-D Echocardiogram   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3   Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4   Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                                                                         transcription  

In [27]:
print("Sloupce Transkripce:", df_en.columns.tolist())
print("Sloupce Registr:", df_cz.columns.tolist())
print("Sloupce OrdinacniHodiny:", df_hours.columns.tolist())
print("Sloupce Sluzby:", df_services.columns.tolist())


Sloupce Transkripce: ['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']
Sloupce Registr: ['MistoPoskytovaniId', 'ZdravotnickeZarizeniId', 'PCZ', 'PCDP', 'NazevCely', 'ZdravotnickeZarizeniKod', 'DruhZarizeniKod', 'DruhZarizeni', 'DruhZarizeniSekundarni', 'Zrizovatel', 'Obec', 'Psc', 'Ulice', 'CisloDomovniOrientacni', 'Kraj', 'KrajKod', 'Okres', 'OkresKod', 'SpravniObvod', 'PoskytovatelTelefon', 'PoskytovatelFax', 'DatumZahajeniCinnosti', 'IdentifikatorDatoveSchranky', 'PoskytovatelEmail', 'PoskytovatelWeb', 'DruhPoskytovatele', 'PoskytovatelNazev', 'Ico', 'TypOsoby', 'PravniFormaKod', 'RUIANKod', 'ORPKod', 'ORP', 'KrajKodSidlo', 'KrajSidlo', 'OkresKodSidlo', 'OkresSidlo', 'PscSidlo', 'ObecSidlo', 'UliceSidlo', 'CisloDomovniOrientacniSidlo', 'OborPece', 'FormaPece', 'DruhPece', 'OdbornyZastupce', 'GPS', 'LastModified']
Sloupce OrdinacniHodiny: ['Ico', 'Kod', 'NazevCely', 'RUIANKod', 'Obec', 'Psc', 'Ulice', 'CisloDomovniOrientacni', 'Okres', 'Kraj', 'CisOddel

In [28]:
# 1. Propojení Registru a Služeb
df_final_cz = pd.merge(df_cz, df_services[['ZdravotnickeZarizeniId', 'PCZ', 'PCDP', 'RozsahPece', 'PoznamkaKeSluzbe']],
                        on=['ZdravotnickeZarizeniId', 'PCZ', 'PCDP'],
                        how='left')

print(f"Česká databáze po propojení: {df_final_cz.shape[0]} záznamů.")
display(df_final_cz.head())

Česká databáze po propojení: 65661 záznamů.


,MistoPoskytovaniId,ZdravotnickeZarizeniId,PCZ,PCDP,NazevCely,ZdravotnickeZarizeniKod,DruhZarizeniKod,DruhZarizeni,DruhZarizeniSekundarni,Zrizovatel,Obec,Psc,Ulice,CisloDomovniOrientacni,Kraj,KrajKod,Okres,OkresKod,SpravniObvod,PoskytovatelTelefon,PoskytovatelFax,DatumZahajeniCinnosti,IdentifikatorDatoveSchranky,PoskytovatelEmail,PoskytovatelWeb,DruhPoskytovatele,PoskytovatelNazev,Ico,TypOsoby,PravniFormaKod,RUIANKod,ORPKod,ORP,KrajKodSidlo,KrajSidlo,OkresKodSidlo,OkresSidlo,PscSidlo,ObecSidlo,UliceSidlo,CisloDomovniOrientacniSidlo,OborPece,FormaPece,DruhPece,OdbornyZastupce,GPS,LastModified,RozsahPece,PoznamkaKeSluzbe
0,261218,181111,1,12,"VIDIA-DIAGNOSTIKA, spol. s r.o., odběrová místnost",41194811001012,362,Odběrová místnost,NaN,NaN,Čerčany,25722.0,Palachova,669,Středočeský kraj,CZ020,Benešov,CZ0201,NaN,+420281911908,NaN,2026-01-30,v99agqp,info@vidia-diagnostika.cz,http://www.vidia-diagnostika.cz,Samostatná odborná laboratoř,"VIDIA-DIAGNOSTIKA, spol. s r.o.",41194811,právnická osoba,Společnost s ručením omezeným,87561832.0,2101,Benešov,CZ010,Hlavní město Praha,CZ0100,Hlavní město Praha,19000.0,Praha 14,Českomoravská,2510/19,"hematologie a transfúzní lékařství, klinická biochemie",ambulantní péče,NaN,MUDr. Marcela Vorlíčková,49.84903867677 14.702378106302,2026-02-01 05:52:59,odběrová místnost,NaN
1,261218,181111,1,12,"VIDIA-DIAGNOSTIKA, spol. s r.o., odběrová místnost",41194811001012,362,Odběrová místnost,NaN,NaN,Čerčany,25722.0,Palachova,669,Středočeský kraj,CZ020,Benešov,CZ0201,NaN,+420281911908,NaN,2026-01-30,v99agqp,info@vidia-diagnostika.cz,http://www.vidia-diagnostika.cz,Samostatná odborná laboratoř,"VIDIA-DIAGNOSTIKA, spol. s r.o.",41194811,právnická osoba,Společnost s ručením omezeným,87561832.0,2101,Benešov,CZ010,Hlavní město Praha,CZ0100,Hlavní město Praha,19000.0,Praha 14,Českomoravská,2510/19,"hematologie a transfúzní lékařství, klinická biochemie",ambulantní péče,NaN,MUDr. Marcela Vorlíčková,49.84903867677 14.702378106302,2026-02-01 05:52:59,odběrová místnost,NaN
2,261212,181110,0,1,MDDr. Blanka Mokošínová,7888791000001,322,Samostatná ordinace PL - stomatologa,NaN,NaN,Brandýs nad Labem-Stará Boleslav,25001.0,Výletní,1411,Středočeský kraj,CZ020,Praha-východ,CZ0209,NaN,+420602448948,NaN,2026-01-29,tu8umr8,zubnipeceboleslav@gmail.com,NaN,Samostatná ordinace PL - stomatologa,MDDr. Blanka Mokošínová,7888791,fyzická osoba,NaN,14624494.0,2103,Brandýs nad Labem-Stará Boleslav,CZ020,Středočeský kraj,CZ0204,Kolín,28171.0,Rostoklaty,NaN,140,zubní lékařství,ambulantní péče,NaN,NaN,50.182852358333 14.656601659872,2026-02-01 05:52:59,NaN,NaN
3,261208,181109,2,0,DKI s.r.o.,24140881002000,302,Poskytovatel amb. služeb (do 5 oborů),Samostatná ordinace PL - stomatologa,NaN,Litoměřice,41201.0,Žitenická,2042,Ústecký kraj,CZ042,Litoměřice,CZ0423,NaN,NaN,NaN,2026-01-29,ehg44fr,NaN,www.stomatologickapece.cz,Samostatná ordinace lékaře specialisty,DKI s.r.o.,24140881,právnická osoba,Společnost s ručením omezeným,2914719.0,4205,Litoměřice,CZ010,Hlavní město Praha,CZ0100,Hlavní město Praha,14700.0,Praha 4,Nad Ryšánkou,2103/7a,"cévní chirurgie, zubní lékařství","ambulantní péče, jednodenní péče",NaN,"Jan Duba, Jan Hrubý, Michal Burian",50.540201912313 14.144876850776,2026-02-01 05:52:59,NaN,NaN
4,261208,181109,2,0,DKI s.r.o.,24140881002000,302,Poskytovatel amb. služeb (do 5 oborů),Samostatná ordinace PL - stomatologa,NaN,Litoměřice,41201.0,Žitenická,2042,Ústecký kraj,CZ042,Litoměřice,CZ0423,NaN,NaN,NaN,2026-01-29,ehg44fr,NaN,www.stomatologickapece.cz,Samostatná ordinace lékaře specialisty,DKI s.r.o.,24140881,právnická osoba,Společnost s ručením omezeným,2914719.0,4205,Litoměřice,CZ010,Hlavní město Praha,CZ0100,Hlavní město Praha,14700.0,Praha 4,Nad Ryšánkou,2103/7a,"cévní chirurgie, zubní lékařství","ambulantní péče, jednodenní péče",NaN,"Jan Duba, Jan Hrubý, Michal Burian",50.540201912313 14.144876850776,2026-02-01 05:52:59,NaN,NaN


In [6]:
print(df_final_cz.columns.tolist())

['MistoPoskytovaniId', 'ZdravotnickeZarizeniId', 'PCZ', 'PCDP', 'NazevCely', 'ZdravotnickeZarizeniKod', 'DruhZarizeniKod', 'DruhZarizeni', 'DruhZarizeniSekundarni', 'Zrizovatel', 'Obec', 'Psc', 'Ulice', 'CisloDomovniOrientacni', 'Kraj', 'KrajKod', 'Okres', 'OkresKod', 'SpravniObvod', 'PoskytovatelTelefon', 'PoskytovatelFax', 'DatumZahajeniCinnosti', 'IdentifikatorDatoveSchranky', 'PoskytovatelEmail', 'PoskytovatelWeb', 'DruhPoskytovatele', 'PoskytovatelNazev', 'Ico', 'TypOsoby', 'PravniFormaKod', 'RUIANKod', 'ORPKod', 'ORP', 'KrajKodSidlo', 'KrajSidlo', 'OkresKodSidlo', 'OkresSidlo', 'PscSidlo', 'ObecSidlo', 'UliceSidlo', 'CisloDomovniOrientacniSidlo', 'OborPece', 'FormaPece', 'DruhPece', 'OdbornyZastupce', 'GPS', 'LastModified', 'RozsahPece', 'PoznamkaKeSluzbe']


In [7]:
# Seznam sloupců, které chceme zachovat
cols_to_keep = [
    'MistoPoskytovaniId', 'Ico', 'DruhZarizeni', 'NazevCely', 'Obec', 'Ulice',
    'CisloDomovniOrientacni', 'Psc', 'Kraj', 'Okres',
    'OborPece', 'GPS', 'PoskytovatelTelefon', 'PoskytovatelEmail',
    'PoskytovatelWeb', 'RozsahPece'
]

# Vytvoření ořezané verze tabulky
df_clean_cz = df_final_cz[cols_to_keep].copy()

# Malý bonus: Spojíme ulici a číslo do jednoho sloupce pro snadnější zobrazení
df_clean_cz['Adresa'] = df_clean_cz['Ulice'] + ' ' + df_clean_cz['CisloDomovniOrientacni']

print(f"Tabulka vyčištěna. Původně sloupců: {df_final_cz.shape[1]}, nyní: {df_clean_cz.shape[1]}")
display(df_clean_cz.tail())

Tabulka vyčištěna. Původně sloupců: 49, nyní: 17


,MistoPoskytovaniId,Ico,DruhZarizeni,NazevCely,Obec,Ulice,CisloDomovniOrientacni,Psc,Kraj,Okres,OborPece,GPS,PoskytovatelTelefon,PoskytovatelEmail,PoskytovatelWeb,RozsahPece,Adresa
65656,225642,582,Lázeňská léčebna,"Vojenská lázeňská a rekreační zařízení, LD Sadový Pramen",Karlovy Vary,Mlýnské nábřeží,574/7,36001.0,Karlovarský kraj,Karlovy Vary,NaN,50.227012053895 12.879344685841,+420973414111,rezervace@je.vlrz.cz,http://www.volareza.cz,NaN,Mlýnské nábřeží 574/7
65657,44171,582,Lázeňská léčebna,"Vojenská lázeňská a rekreační zařízení, LD Kijev",Františkovy Lázně,Národní,15/15,35101.0,Karlovarský kraj,Cheb,NaN,50.118328284076 12.351259519636,+420973414111,rezervace@je.vlrz.cz,http://www.volareza.cz,NaN,Národní 15/15
65658,44170,582,Rehabilitační ústav,"Vojenská lázeňská a rekreační zařízení, Vojenský rehabilitační ústav Slapy nad Vltavou",Slapy,NaN,257,25208.0,Středočeský kraj,Praha-západ,NaN,49.81512701394 14.403865932574,+420973414111,rezervace@je.vlrz.cz,http://www.volareza.cz,NaN,NaN
65659,44170,582,Rehabilitační ústav,"Vojenská lázeňská a rekreační zařízení, Vojenský rehabilitační ústav Slapy nad Vltavou",Slapy,NaN,257,25208.0,Středočeský kraj,Praha-západ,NaN,49.81512701394 14.403865932574,+420973414111,rezervace@je.vlrz.cz,http://www.volareza.cz,NaN,NaN
65660,44170,582,Rehabilitační ústav,"Vojenská lázeňská a rekreační zařízení, Vojenský rehabilitační ústav Slapy nad Vltavou",Slapy,NaN,257,25208.0,Středočeský kraj,Praha-západ,NaN,49.81512701394 14.403865932574,+420973414111,rezervace@je.vlrz.cz,http://www.volareza.cz,NaN,NaN


In [8]:
print(df_en['medical_specialty'].unique())


<StringArray>
[         ' Allergy / Immunology',                    ' Bariatrics',
    ' Cardiovascular / Pulmonary',                     ' Neurology',
                     ' Dentistry',                       ' Urology',
              ' General Medicine',                       ' Surgery',
             ' Speech - Language', ' SOAP / Chart / Progress Notes',
                ' Sleep Medicine',                  ' Rheumatology',
                     ' Radiology',       ' Psychiatry / Psychology',
                      ' Podiatry',     ' Physical Medicine - Rehab',
         ' Pediatrics - Neonatal',               ' Pain Management',
                    ' Orthopedic',                 ' Ophthalmology',
                  ' Office Notes',       ' Obstetrics / Gynecology',
                  ' Neurosurgery',                    ' Nephrology',
                       ' Letters',      ' Lab Medicine - Pathology',
        ' IME-QME-Work Comp etc.',     ' Hospice - Palliative Care',
         ' Hematolog

In [8]:
print(df_final_cz['OborPece'].unique())

['hematologie a transfúzní lékařství, klinická biochemie'
 'zubní lékařství' 'cévní chirurgie, zubní lékařství' ...
 'alergologie a klinická imunologie, dětské lékařství, endokrinologie a diabetologie, klinická biochemie, klinická osteologie, Klinický psycholog, laboratorní a vyšetřovací metody ve zdravotnictví, lékařská genetika, Nutriční terapeut, vnitřní lékařství'
 'alergologie a klinická imunologie, anesteziologie a intenzivní medicína, dětská chirurgie, dětská neurologie, dětské lékařství, endokrinologie a diabetologie, Fyzioterapeut, gynekologie a porodnictví, gynekologie dětí a dospívajících, hematologie a transfúzní lékařství, chirurgie, klinická biochemie, klinická onkologie, Klinický psycholog, laboratorní a vyšetřovací metody ve zdravotnictví, lékařská genetika, nemocniční lékárenství, neonatologie, onkogynekologie, patologie, pediatrie, perinatologie a fetomaternální medicína, plastická chirurgie, Porodní asistentka, radiologie a zobrazovací metody, reprodukční medicína, s

CISTENI TRANSCRIPTIONS

In [10]:
# --- KAGGLE DATA: ZÁKLADNÍ ÚKLID ---
# 1. Odstranění řádků, kde chybí transcription nebo medical_specialty
df_en_clean = df_en.dropna(subset=['transcription', 'medical_specialty']).copy()

# 2. Odstranění identických řádků (surové duplicity v textu)
initial_en = len(df_en_clean)
df_en_clean = df_en_clean.drop_duplicates(subset=['transcription'])

# 3. Odstranění příliš krátkých textů (pod 40 znaků), které nemají význam
df_en_clean = df_en_clean[df_en_clean['transcription'].str.len() > 40]

print(f"Kaggle: Odstraněno {initial_en - len(df_en_clean)} surových duplicit a krátkých textů.")
print(f"Mezisoučet záznamů: {len(df_en_clean)}")

Kaggle: Odstraněno 2617 surových duplicit a krátkých textů.
Mezisoučet záznamů: 2349


In [11]:
import re
import string

def deep_clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()

    # Seznam vzorů pro odstranění (hlavičky, jména, administrativní vata)
    noise_patterns = [
        r'\bsubjective\b:?', r'\bobjective\b:?', r'\bhistory of present illness\b:?',
        r'\bpast medical history\b:?', r'\bmedications\b:?', r'\ballergies\b:?',
        r'\bphysical examination\b:?', r'\bimpression\b:?', r'\bplan\b:?', r'\bassessment\b:?',
        r'\bpreoperative diagnos\w*\b:?', r'\bpostoperative diagnos\w*\b:?',
        r'\bfinal diagnos\w*\b:?', r'\bprocedure\b:?', r'\boperative procedure\b:?',
        r'\bindication\b:?', r'\btechnique\b:?', r'\bname of operation\b:?', r'\banesthesia\b:?',
        r'\breason for visit\b:?', r'\bchief complaint\b:?', r'\bi have seen abc today\b',
        r'\babc\b', r'\bstatus post\b', r'\bdescription\b:?', r'\bsample \w+\b',
        r'\bwifes name\b', r'\bdear sample doctor\b', r'\bmr sample\b',
        r'\bthe patient was placed in the \w+ position\b',
        r'\bprepped and draped in the usual fashion\b',
        r'\bpleasant gentleman\b', r'\bpleasant female\b',
        r'\bcc\b', r'\bhx\b', r'\byo\b', r'\brhf\b',
        r'\bindications\b', r'\bobservations\b', r'\bfindings\b', r'\binterpretation\b',
        r'\bsummary\b', r'\bcomplaints\b', r'\breason for examination\b',
        r'\banatomical summary\b', r'\bexternal examination\b',
        r'\binformed written consent\b', r'\bhas been obtained\b', r'\biexplained the to her\b',
        r'\bthe patient was brought to the\b', r'\bthe body is clad in\b',
        r'\bthis year old\b', r'\bwhite female\b', r'\bafrican american male\b',
        r'\bwho appears the recorded age\b'
    ]

    for pattern in noise_patterns:
        text = re.sub(pattern, ' ', text)

    # Čísla a interpunkce
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))

    # Sjednocení mezer
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [27]:
# Aplikace na dataframe - vytvoření klíčového sloupce pro AI
print("Probíhá hloubkové čištění textů (může to chvíli trvat)...")
df_en_clean['transcription_clean'] = df_en_clean['transcription'].apply(deep_clean_text)

print("Čištění dokončeno. Ukázka vyčištěného textu:")
print(df_en_clean["transcription_clean"].head(3))

Probíhá hloubkové čištění textů (může to chvíli trvat)...
Čištění dokončeno. Ukázka vyčištěného textu:
0    this year old presents with complaint of she used to have when she lived in seattle but she thin...
1    he has difficulty climbing stairs difficulty with airline seats tying shoes used to public seati...
2    he is a very who is years old pounds he is he has a bmi of he has been overweight for ten years ...
Name: transcription_clean, dtype: str


In [28]:
print(df_en_clean.head())

                                                         description  \
0   A 23-year-old white female presents with complaint of allergies.   
1                           Consult for laparoscopic gastric bypass.   
2                           Consult for laparoscopic gastric bypass.   
3                                             2-D M-Mode. Doppler.     
4                                                 2-D Echocardiogram   

            medical_specialty                                sample_name  \
0        Allergy / Immunology                         Allergic Rhinitis    
1                  Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                  Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3  Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4  Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                                                                         transcription  \
0  S

CISTENI REGISTR

In [14]:
# 1. Odstranění prázdných hodnot
df_en_clean = df_en.dropna(subset=['transcription', 'medical_specialty']).copy()

# 2. Odstranění duplikátů
initial_count = len(df_en_clean)
df_en_clean = df_en_clean.drop_duplicates(subset=['transcription'])
print(f"Kaggle: Odstraněno {initial_count - len(df_en_clean)} duplicitních popisů.")

# 3. Odstranění příliš krátkých textů (např. pod 50 znaků), které nemají výpovědní hodnotu
df_en_clean = df_en_clean[df_en_clean['transcription'].str.len() > 40]

print(f"Kaggle: Konečný počet vzorků pro trénování: {len(df_en_clean)}")


Kaggle: Odstraněno 2609 duplicitních popisů.
Kaggle: Konečný počet vzorků pro trénování: 2349


In [9]:
# 1. Normalizace textů v OborPece
df_clean_cz['OborPece'] = df_clean_cz['OborPece'].str.lower()

# 2. Vyčištění GPS
# Předpokládáme, že GPS je ve formátu "50.123, 14.456"
df_clean_cz = df_clean_cz[df_clean_cz['GPS'].notna() & (df_clean_cz['GPS'] != '0.0 0.0')]

# 3. Odstranění totálních duplicit
df_clean_cz = df_clean_cz.drop_duplicates()

# 4. Bonus: Vyčištění telefonů (odstranění mezer a divných znaků)
df_clean_cz['PoskytovatelTelefon'] = df_clean_cz['PoskytovatelTelefon'].str.replace(r'\s+', '', regex=True)

print(f"Česká data: Konečný počet záznamů po očištění: {len(df_clean_cz)}")

Česká data: Konečný počet záznamů po očištění: 42158


In [10]:
# Jen pro tvou informaci: Co dělají ti bez oboru?
print(df_final_cz[df_final_cz['OborPece'].isna()]['DruhZarizeni'].value_counts().head(10))

DruhZarizeni
Zdravotní péče v ústavech sociální p.    456
Zdravotnická zachranná služba            445
Lékárna                                  365
Zdravotnická dopravní služba             214
Výdejna zdravotnických prostředků         92
Lázeňský dům (jen ubyt. hotel. typu)      85
Přeprava pacientů neodkladné péče         70
Domácí zdravotní péče                     66
Léčebna pro dlouhodobě nemocné (LDN)      26
Odběrová místnost                         19
Name: count, dtype: int64


In [11]:
# 1. Převod IČO na string a doplnění nul na začátek
df_clean_cz['Ico'] = df_clean_cz['Ico'].astype(str).str.zfill(8)

# 2. ODSTRANĚNÍ ŘÁDKŮ bez Oboru Péče
# Tímto se zbavíme lékáren, sanitek a dalších věcí, které pro AI klasifikaci nepotřebujeme
initial_count = len(df_clean_cz)
df_clean_cz = df_clean_cz.dropna(subset=['OborPece']).copy()

# 3. Kontrola chybějících hodnot po pročištění
print(f"Čištění dokončeno. Odstraněno {initial_count - len(df_clean_cz)} záznamů (lékárny, sanitky atd.).")
print("\nZbývající chybějící hodnoty v kritických sloupcích:")
print(df_clean_cz[['NazevCely', 'OborPece', 'Obec']].isna().sum())

Čištění dokončeno. Odstraněno 1715 záznamů (lékárny, sanitky atd.).

Zbývající chybějící hodnoty v kritických sloupcích:
NazevCely    0
OborPece     0
Obec         0
dtype: int64


In [12]:
# 1. Rozdělíme řetězec podle čárky na skutečný seznam (list)
df_clean_cz['OborPece_List'] = df_clean_cz['OborPece'].str.split(',')

# 2. Funkce explode "rozbalí" seznam do samostatných řádků
df_expanded_cz = df_clean_cz.explode('OborPece_List')

# 3. Očistíme obory od mezer a převedeme na malá písmena
df_expanded_cz['OborPece_List'] = df_expanded_cz['OborPece_List'].str.strip().str.lower()

print(f"Původní počet řádků: {len(df_clean_cz)}")
print(f"Nový počet řádků po rozbalení oborů: {len(df_expanded_cz)}")

# Ukázka, jak to teď vypadá
display(df_expanded_cz.head(15))

Původní počet řádků: 40443
Nový počet řádků po rozbalení oborů: 76321


,MistoPoskytovaniId,Ico,DruhZarizeni,NazevCely,Obec,Ulice,CisloDomovniOrientacni,Psc,Kraj,Okres,OborPece,GPS,PoskytovatelTelefon,PoskytovatelEmail,PoskytovatelWeb,RozsahPece,Adresa,OborPece_List
0,261218,41194811,Odběrová místnost,"VIDIA-DIAGNOSTIKA, spol. s r.o., odběrová místnost",Čerčany,Palachova,669,25722.0,Středočeský kraj,Benešov,"hematologie a transfúzní lékařství, klinická biochemie",49.84903867677 14.702378106302,+420281911908,info@vidia-diagnostika.cz,http://www.vidia-diagnostika.cz,odběrová místnost,Palachova 669,hematologie a transfúzní lékařství
0,261218,41194811,Odběrová místnost,"VIDIA-DIAGNOSTIKA, spol. s r.o., odběrová místnost",Čerčany,Palachova,669,25722.0,Středočeský kraj,Benešov,"hematologie a transfúzní lékařství, klinická biochemie",49.84903867677 14.702378106302,+420281911908,info@vidia-diagnostika.cz,http://www.vidia-diagnostika.cz,odběrová místnost,Palachova 669,klinická biochemie
2,261212,07888791,Samostatná ordinace PL - stomatologa,MDDr. Blanka Mokošínová,Brandýs nad Labem-Stará Boleslav,Výletní,1411,25001.0,Středočeský kraj,Praha-východ,zubní lékařství,50.182852358333 14.656601659872,+420602448948,zubnipeceboleslav@gmail.com,NaN,NaN,Výletní 1411,zubní lékařství
3,261208,24140881,Poskytovatel amb. služeb (do 5 oborů),DKI s.r.o.,Litoměřice,Žitenická,2042,41201.0,Ústecký kraj,Litoměřice,"cévní chirurgie, zubní lékařství",50.540201912313 14.144876850776,NaN,NaN,www.stomatologickapece.cz,NaN,Žitenická 2042,cévní chirurgie
3,261208,24140881,Poskytovatel amb. služeb (do 5 oborů),DKI s.r.o.,Litoměřice,Žitenická,2042,41201.0,Ústecký kraj,Litoměřice,"cévní chirurgie, zubní lékařství",50.540201912313 14.144876850776,NaN,NaN,www.stomatologickapece.cz,NaN,Žitenická 2042,zubní lékařství
6,261204,03582485,Samostatná ordinace lékaře specialisty,Neo Surgery Services s.r.o.,Olomouc,Wolkerova,656/22,77900.0,Olomoucký kraj,Olomouc,plastická chirurgie,49.587903029498 17.243683375997,NaN,NaN,NaN,NaN,Wolkerova 656/22,plastická chirurgie
7,261201,24426610,Samostatná ordinace lékaře specialisty,MUDr. Tomáš Kovanda,Brno,Bulharská,1320/29,61200.0,Jihomoravský kraj,Brno-město,ortopedie a traumatologie pohybového ústrojí,49.226060532661 16.590980688573,NaN,NaN,NaN,NaN,Bulharská 1320/29,ortopedie a traumatologie pohybového ústrojí
8,261198,23741431,Oční optika,OPTIKA MAON Vyškov s.r.o.,Vyškov,Purkyňova,235/36,68201.0,Jihomoravský kraj,Vyškov,optometrista,49.27751356829 16.979920879488,NaN,NaN,NaN,NaN,Purkyňova 235/36,optometrista
9,261195,24419907,Samostatná ordinace lékaře specialisty,MUDr. Monika Trnová Chudárková,Brno,Viniční,4049/235,61500.0,Jihomoravský kraj,Brno-město,cévní chirurgie,49.198676189054 16.656428622322,NaN,NaN,NaN,NaN,Viniční 4049/235,cévní chirurgie
10,261186,21480087,Samostatné zařízení fyzioterapeuta,Mgr. KATEŘINA KOROŠOVÁ,Veltrusy,Palackého,117,27746.0,Středočeský kraj,Mělník,fyzioterapeut,50.270145700813 14.329106241488,NaN,NaN,NaN,individuální fyzioterapie,Palackého 117,fyzioterapeut


In [13]:
# Kolik máme unikátních zařízení?
unique_clinics = df_expanded_cz['MistoPoskytovaniId'].nunique()
print(f"Máme {unique_clinics} unikátních ordinací připravených k doporučení.")

Máme 38889 unikátních ordinací připravených k doporučení.


In [14]:
# Které obory jsou nejčastější?
print(df_expanded_cz['OborPece_List'].value_counts().head(65))

OborPece_List
všeobecné praktické lékařství       6595
zubní lékařství                     6058
fyzioterapeut                       4698
vnitřní lékařství                   2772
praktické lékárenství               2525
                                    ... 
gynekologie dětí a dospívajících     199
pediatrie                            191
nukleární medicína                   188
radiační onkologie                   183
klinická osteologie                  176
Name: count, Length: 65, dtype: int64


In [21]:
# Které obory jsou nejčastější?
print(df_en_clean['medical_specialty'].value_counts().head(30))

medical_specialty
Surgery                          974
Radiology                        247
General Medicine                 157
Urology                          155
SOAP / Chart / Progress Notes    143
Neurology                         67
Consult - History and Phy.        55
Orthopedic                        54
Pediatrics - Neonatal             52
Psychiatry / Psychology           51
Pain Management                   44
Office Notes                      41
Hematology - Oncology             31
Gastroenterology                  31
Cardiovascular / Pulmonary        26
Obstetrics / Gynecology           26
Nephrology                        20
Sleep Medicine                    17
Emergency Room Reports            17
Discharge Summary                 17
ENT - Otolaryngology              16
Ophthalmology                     15
Bariatrics                        10
Podiatry                          10
Physical Medicine - Rehab         10
Speech - Language                  9
IME-QME-Work Comp et

In [15]:
# 3. Kontrola kritických polí pro aplikaci
print("\nČeský registr - Chybějící hodnoty:")
# Zajímá nás hlavně název a obor, zbytek (web, email) může chybět, to přežijeme
print(df_expanded_cz[['NazevCely', 'OborPece_List', 'Obec']].isnull().sum())


Český registr - Chybějící hodnoty:
NazevCely        0
OborPece_List    0
Obec             0
dtype: int64


MAPPING

In [19]:
medical_mapping = {
    "Surgery": ["chirurgie", "dětská chirurgie", "cévní chirurgie", "traumatologie", "plastická chirurgie"],
    "Cardiovascular / Pulmonary": ["kardiologie", "dětská kardiologie", "angiologie", "pneumologie a ftizeologie", "vnitřní lékařství"],
    "Orthopedic": ["ortopedie a traumatologie pohybového ústrojí", "fyzioterapeut", "rehabilitační a fyzikální medicína", "ergoterapeut"],
    "Radiology": ["radiologie a zobrazovací metody"],
    "General Medicine": ["všeobecné praktické lékařství", "vnitřní lékařství", "praktické lékařství pro děti a dorost"],
    "Gastroenterology": ["gastroenterologie"],
    "Neurology": ["neurologie", "dětská neurologie"],
    "Obstetrics / Gynecology": ["gynekologie a porodnictví", "porodní asistentka"],
    "Urology": ["urologie"],
    "ENT - Otolaryngology": ["otorinolaryngologie a chirurgie hlavy a krku", "foniatrie"],
    "Neurosurgery": ["neurologie", "chirurgie"],
    "Hematology - Oncology": ["hematologie a transfúzní lékařství", "klinická onkologie"],
    "Ophthalmology": ["oftalmologie", "optometrista"],
    "Nephrology": ["nefrologie"],
    "Pediatrics - Neonatal": ["praktické lékařství pro děti a dorost", "dětské lékařství", "neonatologie"],
    "Pain Management": ["algeziologie", "rehabilitační a fyzikální medicína"],
    "Psychiatry / Psychology": ["psychiatrie", "klinický psycholog", "dětská a dorostová psychiatrie"],
    "Dermatology": ["dermatovenerologie"],
    "Dentistry": ["zubní lékařství", "dentální hygienistka", "ortodoncie"],
    "Endocrinology": ["endokrinologie a diabetologie"],
    "Physical Medicine - Rehab": ["rehabilitační a fyzikální medicína", "fyzioterapeut"],
    "Allergy / Immunology": ["alergologie a klinická imunologie"]
}

In [29]:
# --- 2. VYČIŠTĚNÍ MAPPINGU (Normalizace) ---
# Ošetříme případné mezery na začátku/konci, aby filtrace stoprocentně seděla
clean_mapping = {}
for en_key, cz_list in medical_mapping.items():
    new_key = en_key.strip()
    new_vals = [v.strip().lower() for v in cz_list]
    clean_mapping[new_key] = new_vals

# --- 3. ÚKLID V DATAFRAMECH ---
# Odstraníme neviditelné mezery v názvech kategorií v obou tabulkách
df_en_clean['medical_specialty'] = df_en_clean['medical_specialty'].str.strip()
df_expanded_cz['OborPece_List'] = df_expanded_cz['OborPece_List'].str.strip().str.lower()

# --- 4. FINÁLNÍ FILTRACE KAGGLE DAT ---
# Ponecháme jen ty záznamy, pro které máme český ekvivalent v mappingu
supported_specialties = list(clean_mapping.keys())
df_en_final = df_en_clean[df_en_clean['medical_specialty'].isin(supported_specialties)].copy()

# --- 5. VÝSLEDEK ---
print(f"--- REKAPITULACE FILTRACE ---")
print(f"Původní počet unikátních anglických záznamů: {len(df_en_clean)}")
print(f"Počet záznamů po filtraci na podporované obory: {len(df_en_final)}")
print(f"Počet unikátních kategorií pro AI trénování: {df_en_final['medical_specialty'].nunique()}")

if len(df_en_final) > 0:
    print("\n✅ Úspěch! Dataset je připraven k uložení a trénování.")
else:
    print("\n❌ CHYBA: Po filtraci nezůstala žádná data. Zkontroluj názvy kategorií.")

--- REKAPITULACE FILTRACE ---
Původní počet unikátních anglických záznamů: 2349
Počet záznamů po filtraci na podporované obory: 1984
Počet unikátních kategorií pro AI trénování: 22

✅ Úspěch! Dataset je připraven k uložení a trénování.


EXPORT

In [20]:
import os
import joblib

# 1. Vytvoření cesty k adresáři (pokud neexistuje)
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# 2. Uložení vyčištěného Kaggle datasetu (1 985 záznamů)
kaggle_path = os.path.join(output_dir, 'kaggle_final.csv')
df_en_final.to_csv(kaggle_path, index=False, encoding='utf-8')

# 3. Uložení expandovaného českého registru
registr_path = os.path.join(output_dir, 'czech_registre_final.csv')
df_expanded_cz.to_csv(registr_path, index=False, encoding='utf-8')

# 4. Uložení mapping slovníku (budeme ho potřebovat pro propojení AI s vyhledáváním)
mapping_path = os.path.join(output_dir, 'medical_mapping.pkl')
joblib.dump(medical_mapping, mapping_path)

print("--- EXPORT DOKONČEN ---")
print(f"1. Kaggle uložen do: {kaggle_path}")
print(f"2. Registr uložen do: {registr_path}")
print(f"3. Mapping uložen do: {mapping_path}")
print("-----------------------")
print("Vše je připraveno pro Fázi 2: Trénování modelu.")

--- EXPORT DOKONČEN ---
1. Kaggle uložen do: ../data/processed\kaggle_final.csv
2. Registr uložen do: ../data/processed\czech_registre_final.csv
3. Mapping uložen do: ../data/processed\medical_mapping.pkl
-----------------------
Vše je připraveno pro Fázi 2: Trénování modelu.


In [22]:
# Uložení do formátu Parquet (bude mít odhadem jen 3-5 MB)
registr_path_parquet = os.path.join(output_dir, 'czech_registre_final.parquet')
df_expanded_cz.to_parquet(registr_path_parquet, index=False)

In [23]:
# Necháme jen to, co uživatel vidí na webu
potrebne_sloupce = [
    'NazevCely', 'Obec', 'Ulice', 'OborPece_List',
    'GPS', 'PoskytovatelTelefon', 'PoskytovatelEmail', 'PoskytovatelWeb'
]
df_expanded_cz_final= df_expanded_cz[potrebne_sloupce]
df_expanded_cz_final.to_csv('../data/processed/czech_registre_final_small.csv', index=False)